# Projet 1 - Reconnaissance faciale des étudiants (OpenCV LBPH)
**Cours :** ANI-2083 - ENSPY &nbsp;|&nbsp; **Enseignant :** Dr. Aurelle TCHAGNA

**Groupe :** *(à compléter : noms des 5 membres)* &nbsp;|&nbsp; **Date :** *(à compléter)*

## Objectif
Détecter un visage, identifier l'étudiant parmi 5 personnes enregistrées, afficher un score, et répondre **« Inconnu »** si la confiance est insuffisante.

## Méthode
- **Détection :** Haar Cascade d'OpenCV.
- **Reconnaissance :** LBPH (Local Binary Patterns Histograms), qui renvoie la personne la plus proche et un **score de distance** (plus il est petit, plus la ressemblance est forte).
- **Inconnu :** si la distance dépasse un **seuil**, le visage est déclaré inconnu.

Les photos ont été prises avec l'accord des personnes, pour un usage strictement pédagogique.

## 1. Imports et installation

In [4]:
# Dans Google Colab, décommentez la ligne suivante (une seule fois) :
# !pip install opencv-contrib-python

import os, json, random
import cv2
import numpy as np
import matplotlib.pyplot as plt

# LBPH se trouve dans le module cv2.face (fourni par opencv-contrib-python)
print("OpenCV version :", cv2.__version__)
print("LBPH disponible :", hasattr(cv2, "face"))

OpenCV version : 5.0.0
LBPH disponible : True


## 2. Configuration
Le dossier de données doit contenir un sous-dossier par personne, plus un dossier `Inconnu` (personne **absente** de la base, utilisée **uniquement pour le test**) :
```
data/raw/Marie/  data/raw/Paul/  ...  data/raw/Inconnu/
```
Sous Colab, mettez `data/` sur Google Drive et décommentez le montage.

In [5]:
# --- Sous Google Colab : décommentez ces 2 lignes ---
# from google.colab import drive
# drive.mount('/content/drive')

DOSSIER_DATA = "data/raw"          # Colab : "/content/drive/MyDrive/projet1/data/raw"
DOSSIER_MODELES = "models"
NOM_INCONNU = "Inconnu"            # dossier de la personne absente de la base
TAILLE = 200                       # taille des visages recadrés (200x200)
PART_TEST = 0.2                    # 20 % des photos gardées pour le test
GRAINE = 42                        # pour obtenir toujours le même découpage

os.makedirs(DOSSIER_MODELES, exist_ok=True)
random.seed(GRAINE)

## 3. Chargement du dataset
On compte les photos de chaque personne. Ce tableau ira dans le rapport.

In [6]:
EXTENSIONS = (".jpg", ".jpeg", ".png")

def lister_photos(dossier):
    return sorted(os.path.join(dossier, f) for f in os.listdir(dossier)
                  if f.lower().endswith(EXTENSIONS))

noms = sorted(d for d in os.listdir(DOSSIER_DATA)
              if os.path.isdir(os.path.join(DOSSIER_DATA, d)) and d != NOM_INCONNU)

photos = {nom: lister_photos(os.path.join(DOSSIER_DATA, nom)) for nom in noms}

print("Personnes enregistrées :", noms)
for nom in noms:
    print(f"  {nom:15s} {len(photos[nom])} photos")
print("Total :", sum(len(v) for v in photos.values()), "photos")

chemin_inconnu = os.path.join(DOSSIER_DATA, NOM_INCONNU)
if os.path.isdir(chemin_inconnu):
    print(f"Photos de test '{NOM_INCONNU}' :", len(lister_photos(chemin_inconnu)))
else:
    print(f"ATTENTION : dossier '{NOM_INCONNU}' introuvable (nécessaire pour l'étape 9).")

FileNotFoundError: [WinError 3] Le chemin d’accès spécifié est introuvable: 'data/raw'

## 4. Aperçu des données
On affiche 4 photos par personne pour vérifier visuellement la qualité.

In [ ]:
fig, axes = plt.subplots(len(noms), 4, figsize=(10, 2.4 * len(noms)))
axes = np.atleast_2d(axes)
for i, nom in enumerate(noms):
    for j in range(4):
        axes[i, j].axis("off")
        if j < len(photos[nom]):
            img = cv2.imread(photos[nom][j])
            if img is not None:
                axes[i, j].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if j == 0:
            axes[i, j].set_title(nom, loc="left")
plt.tight_layout()
plt.show()

## 5. Détection et recadrage des visages
Pour chaque photo : détection du visage (Haar Cascade), recadrage, passage en gris, redimensionnement en 200×200.
Les photos où **aucun visage n'est détecté** sont comptées (chiffre à citer dans le rapport).

In [ ]:
detecteur = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def extraire_visage(chemin):
    """Renvoie le visage recadré (gris, TAILLE x TAILLE) ou None si aucun visage."""
    image = cv2.imread(chemin)
    if image is None:
        return None
    gris = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Réduire les grandes photos de téléphone pour aller plus vite
    if max(gris.shape) > 1000:
        echelle = 1000 / max(gris.shape)
        gris = cv2.resize(gris, None, fx=echelle, fy=echelle)
    visages = detecteur.detectMultiScale(gris, scaleFactor=1.2, minNeighbors=5, minSize=(60, 60))
    if len(visages) == 0:
        return None
    # Si plusieurs visages, garder le plus grand
    x, y, w, h = max(visages, key=lambda v: v[2] * v[3])
    visage = cv2.resize(gris[y:y+h, x:x+w], (TAILLE, TAILLE))
    return cv2.equalizeHist(visage)   # égalise la lumière

donnees = {}      # nom -> liste de (visage, chemin)
sans_visage = {}  # nom -> nombre de photos sans visage
for nom in noms:
    donnees[nom], sans_visage[nom] = [], 0
    for chemin in photos[nom]:
        v = extraire_visage(chemin)
        if v is None:
            sans_visage[nom] += 1
        else:
            donnees[nom].append((v, chemin))

print("Visages détectés / photos :")
for nom in noms:
    print(f"  {nom:15s} {len(donnees[nom])} / {len(photos[nom])}  (sans visage : {sans_visage[nom]})")

## 6. Séparation entraînement / test
80 % des visages de chaque personne servent à entraîner, 20 % à tester.

In [ ]:
etiquette_de = {nom: i for i, nom in enumerate(noms)}       # nom -> numéro
nom_de = {i: nom for nom, i in etiquette_de.items()}         # numéro -> nom

train, test = [], []        # listes de (visage, numéro, chemin)
for nom in noms:
    liste = donnees[nom][:]
    random.shuffle(liste)
    n_test = max(1, round(PART_TEST * len(liste)))
    for k, (v, chemin) in enumerate(liste):
        (test if k < n_test else train).append((v, etiquette_de[nom], chemin))

print("Photos d'entraînement :", len(train))
print("Photos de test        :", len(test))

## 7. Entraînement du modèle LBPH
Le modèle et le dictionnaire des noms sont sauvegardés dans `models/`.

In [ ]:
modele = cv2.face.LBPHFaceRecognizer_create()
modele.train([t[0] for t in train], np.array([t[1] for t in train]))

modele.save(os.path.join(DOSSIER_MODELES, "modele_lbph.yml"))
with open(os.path.join(DOSSIER_MODELES, "noms.json"), "w", encoding="utf-8") as f:
    json.dump(nom_de, f, ensure_ascii=False)

print("Modèle entraîné et sauvegardé dans", DOSSIER_MODELES)
print("Correspondance numéro -> nom :", nom_de)

## 8. Évaluation sur les photos de test (personnes connues)
On calcule la précision et on affiche les erreurs éventuelles.

In [ ]:
resultats = []   # (vrai, prédit, distance, visage)
for v, vrai, chemin in test:
    predit, distance = modele.predict(v)
    resultats.append((vrai, predit, distance, v))

justes = sum(1 for r in resultats if r[0] == r[1])
precision = justes / len(resultats)
print(f"Précision : {justes}/{len(resultats)} = {precision:.1%}")

# Matrice de confusion (lignes = vraie personne, colonnes = prédite)
n = len(noms)
matrice = np.zeros((n, n), dtype=int)
for vrai, predit, _, _ in resultats:
    matrice[vrai, predit] += 1
print("\nMatrice de confusion (lignes = vrai, colonnes = prédit) :")
print("Ordre :", [nom_de[i] for i in range(n)])
print(matrice)

# Exemples d'erreurs
erreurs = [r for r in resultats if r[0] != r[1]][:5]
if erreurs:
    fig, axes = plt.subplots(1, len(erreurs), figsize=(3 * len(erreurs), 3))
    axes = np.atleast_1d(axes)
    for ax, (vrai, predit, dist, v) in zip(axes, erreurs):
        ax.imshow(v, cmap="gray"); ax.axis("off")
        ax.set_title(f"vrai : {nom_de[vrai]}\nprédit : {nom_de[predit]}\ndist. {dist:.0f}", fontsize=9)
    plt.show()
else:
    print("Aucune erreur sur les photos de test.")

**Commentaire (à compléter) :** *La précision est de ... %. Les erreurs viennent surtout de ... (éclairage, angle, distance...).*

> Attention : les photos d'une même séance se ressemblent beaucoup, la précision peut donc paraître meilleure qu'en situation réelle. Le test « Inconnu » ci-dessous est plus exigeant.

## 9. Test « Inconnu »
On fait reconnaître les photos de la personne **absente de la base**. Le modèle va forcément proposer quelqu'un, mais la **distance** devrait être plus grande.

In [ ]:
scores_inconnu = []
for chemin in lister_photos(chemin_inconnu):
    v = extraire_visage(chemin)
    if v is not None:
        _, distance = modele.predict(v)
        scores_inconnu.append(distance)

scores_connu = [r[2] for r in resultats if r[0] == r[1]]   # distances des bonnes reconnaissances

print(f"Personnes connues (bien reconnues) : {len(scores_connu)} scores")
print(f"  min {min(scores_connu):.0f} | médiane {np.median(scores_connu):.0f} | max {max(scores_connu):.0f}")
print(f"Personne inconnue : {len(scores_inconnu)} scores")
print(f"  min {min(scores_inconnu):.0f} | médiane {np.median(scores_inconnu):.0f} | max {max(scores_inconnu):.0f}")

## 10. Choix du seuil
Au-dessus du seuil, l'application affiche **« Inconnu »**. On regarde où se séparent les deux groupes de scores, puis **on choisit la valeur nous-mêmes**.

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.hist(scores_connu, bins=20, alpha=0.7, label="Connus (bien reconnus)")
plt.hist(scores_inconnu, bins=20, alpha=0.7, label="Inconnu")
plt.xlabel("Distance LBPH (petit = ressemblant)")
plt.ylabel("Nombre de photos")
plt.legend()
plt.show()

# Suggestion automatique : milieu entre le haut des connus et le bas des inconnus
suggestion = (np.percentile(scores_connu, 95) + np.percentile(scores_inconnu, 5)) / 2
print(f"Suggestion de départ : {suggestion:.0f}")

In [ ]:
SEUIL = 80   # <-- À MODIFIER après avoir regardé le graphique (voir suggestion ci-dessus)

connus_acceptes = np.mean([s <= SEUIL for s in scores_connu])
inconnus_rejetes = np.mean([s > SEUIL for s in scores_inconnu])
print(f"Seuil = {SEUIL}")
print(f"  Connus reconnus (distance <= seuil)  : {connus_acceptes:.1%}")
print(f"  Inconnus rejetés (distance > seuil)  : {inconnus_rejetes:.1%}")

# Sauvegarde du seuil pour l'application webcam
with open(os.path.join(DOSSIER_MODELES, "config.json"), "w") as f:
    json.dump({"seuil": SEUIL, "taille": TAILLE}, f)
print("Seuil sauvegardé dans models/config.json")

**Commentaire (à compléter) :** *J'ai choisi le seuil ... parce que ... Avec ce seuil, ... % des connus sont reconnus et ... % des inconnus sont rejetés.*

Compromis : un seuil **bas** rejette plus d'inconnus mais refuse aussi des personnes connues ; un seuil **haut** fait l'inverse.

## 11. Conclusion
**À compléter :**
- Résultats : précision ..., seuil ..., inconnus rejetés ...
- Limites : peu de photos, sensibilité à la lumière et à l'angle, LBPH moins précis que les méthodes modernes.
- Améliorations possibles : plus de photos variées, autre méthode (embeddings), meilleure détection.